# 🧩 Cryptarithmetic Puzzle Solver: SEND + MORE = MONEY
## Constraint Satisfaction Problem (CSP) Approach
**Course**: Artificial Intelligence Lab Assessment  
**Author**: Chirag Kankani  

---

### Problem Statement:
Find distinct single-digit integer assignments (0–9) for each letter in the equation:
$$\text{SEND} + \text{MORE} = \text{MONEY}$$
Such that:
1. Each letter represents a unique digit (**AllDifferent** constraint).
2. Leading digits cannot be zero ($S \ne 0$, $M \ne 0$).
3. The arithmetic sum is strictly satisfied.

### AI Techniques Implemented & Evaluated in this Notebook:
- **Step 1**: CSP Problem Formulation (Variables, Domains, Constraints)
- **Step 2**: Column-wise Arithmetic Consistency Checks
- **Step 3**: Pure Backtracking Search (Baseline)
- **Step 4**: Backtracking with Forward Checking (FC)
- **Step 5**: Minimum Remaining Values (MRV) Variable Selection Heuristic
- **Step 6**: Least Constraining Value (LCV) Value Ordering Heuristic
- **Step 7**: AC-3 Constraint Propagation (Arc Consistency & Domain Reduction)
- **Step 8**: Comprehensive Comparative Evaluation Table
- **Step 9**: Solution Display & Verification
- **Step 10**: Viva Voce Q&A and Professor Explanation Guide


In [ ]:
# =====================================================================
# STEP 1: Define Variables, Domains, and Constraints
# =====================================================================

VARIABLES = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']

def get_initial_domains():
    """
    Initial Domains:
    - S, M in {1..9} because leading digits cannot be zero.
    - E, N, D, O, R, Y in {0..9}.
    """
    domains = {}
    for var in VARIABLES:
        if var in ['S', 'M']:
            domains[var] = list(range(1, 10)) # No leading zeros!
        else:
            domains[var] = list(range(0, 10))
    return domains

print("Variables:", VARIABLES)
print("\nInitial Domains:")
for k, v in get_initial_domains().items():
    print(f"  Domain({k}) = {v}")


In [ ]:
# =====================================================================
# STEP 2: Arithmetic Column Constraints & Full Verification
# =====================================================================

def check_partial_arithmetic(assignment):
    """
    Column-wise arithmetic checks from right to left:
      Col 0 (Units):     D + E = Y + 10 * C1
      Col 1 (Tens):      N + R + C1 = E + 10 * C2
      Col 2 (Hundreds):  E + O + C2 = N + 10 * C3
      Col 3 (Thousands): S + M + C3 = O + 10 * C4
      Col 4 (Ten-Thous): C4 = M
    """
    if assignment.get('S') == 0 or assignment.get('M') == 0:
        return False

    # Units column (D, E, Y)
    if 'D' in assignment and 'E' in assignment and 'Y' in assignment:
        if (assignment['D'] + assignment['E']) % 10 != assignment['Y']:
            return False

    # Tens column (N, R, E)
    if 'D' in assignment and 'E' in assignment and 'N' in assignment and 'R' in assignment:
        c1 = (assignment['D'] + assignment['E']) // 10
        if (assignment['N'] + assignment['R'] + c1) % 10 != assignment['E']:
            return False

    # Hundreds column (E, O, N)
    if 'D' in assignment and 'E' in assignment and 'N' in assignment and 'R' in assignment and 'O' in assignment:
        c1 = (assignment['D'] + assignment['E']) // 10
        c2 = (assignment['N'] + assignment['R'] + c1) // 10
        if (assignment['E'] + assignment['O'] + c2) % 10 != assignment['N']:
            return False

    # Thousands & Ten-Thousands columns (S, M, O)
    if ('D' in assignment and 'E' in assignment and 'N' in assignment and 'R' in assignment 
            and 'O' in assignment and 'S' in assignment and 'M' in assignment):
        c1 = (assignment['D'] + assignment['E']) // 10
        c2 = (assignment['N'] + assignment['R'] + c1) // 10
        c3 = (assignment['E'] + assignment['O'] + c2) // 10
        if (assignment['S'] + assignment['M'] + c3) % 10 != assignment['O']:
            return False
        c4 = (assignment['S'] + assignment['M'] + c3) // 10
        if c4 != assignment['M']:
            return False

    return True

def verify_solution(assignment):
    """Verifies the full mathematical equation."""
    if len(assignment) != 8:
        return False
    if len(set(assignment.values())) != 8:
        return False
    s, e, n, d = assignment['S'], assignment['E'], assignment['N'], assignment['D']
    m, o, r, y = assignment['M'], assignment['O'], assignment['R'], assignment['Y']
    send = 1000*s + 100*e + 10*n + d
    more = 1000*m + 100*o + 10*r + e
    money = 10000*m + 1000*o + 100*n + 10*e + y
    return send + more == money

print("Constraint verification functions defined successfully!")


In [ ]:
# =====================================================================
# STEP 3: Algorithm 1 - Pure Backtracking Search (Baseline)
# =====================================================================
import time

def solve_pure_backtracking():
    domains = get_initial_domains()
    column_order = ['D', 'E', 'Y', 'N', 'R', 'O', 'S', 'M']
    nodes = 0
    backtracks = 0
    t0 = time.perf_counter()

    def backtrack(assignment):
        nonlocal nodes, backtracks
        nodes += 1
        if len(assignment) == len(VARIABLES):
            return assignment if verify_solution(assignment) else None

        # Chronological selection
        var = next(v for v in column_order if v not in assignment)

        for val in domains[var]:
            # AllDifferent check
            if val in assignment.values():
                continue
            new_assign = dict(assignment)
            new_assign[var] = val

            if not check_partial_arithmetic(new_assign):
                continue

            res = backtrack(new_assign)
            if res is not None:
                return res
            backtracks += 1
        return None

    ans = backtrack({})
    elapsed = (time.perf_counter() - t0) * 1000.0
    return ans, nodes, backtracks, elapsed

sol_bt, nodes_bt, bts_bt, el_bt = solve_pure_backtracking()
print(f"[Pure Backtracking] Result: {sol_bt}")
print(f"Nodes Explored: {nodes_bt} | Backtracks: {bts_bt} | Time: {el_bt:.2f} ms")


In [ ]:
# =====================================================================
# STEP 4: Algorithm 2 - Backtracking + Forward Checking (FC)
# =====================================================================

def forward_check(var, val, assignment, domains):
    """Prunes val from all other unassigned variables sharing AllDifferent."""
    new_domains = {}
    for other_var, other_vals in domains.items():
        if other_var not in assignment and other_var != var:
            pruned = [v for v in other_vals if v != val]
            if len(pruned) == 0:
                return None, False # Domain wipeout! Prune branch early
            new_domains[other_var] = pruned
        else:
            new_domains[other_var] = list(other_vals)
    return new_domains, True

def solve_forward_checking():
    domains = get_initial_domains()
    column_order = ['D', 'E', 'Y', 'N', 'R', 'O', 'S', 'M']
    nodes = 0
    backtracks = 0
    t0 = time.perf_counter()

    def backtrack(assignment, cur_domains):
        nonlocal nodes, backtracks
        nodes += 1
        if len(assignment) == len(VARIABLES):
            return assignment if verify_solution(assignment) else None

        var = next(v for v in column_order if v not in assignment)

        for val in cur_domains[var]:
            if val in assignment.values():
                continue
            new_assign = dict(assignment)
            new_assign[var] = val

            if not check_partial_arithmetic(new_assign):
                continue

            # Forward Checking Inference
            pruned_domains, success = forward_check(var, val, new_assign, cur_domains)
            if not success:
                backtracks += 1
                continue

            res = backtrack(new_assign, pruned_domains)
            if res is not None:
                return res
            backtracks += 1
        return None

    ans = backtrack({}, domains)
    elapsed = (time.perf_counter() - t0) * 1000.0
    return ans, nodes, backtracks, elapsed

sol_fc, nodes_fc, bts_fc, el_fc = solve_forward_checking()
print(f"[Backtracking + Forward Checking] Result: {sol_fc}")
print(f"Nodes Explored: {nodes_fc} | Backtracks: {bts_fc} | Time: {el_fc:.2f} ms")


In [ ]:
# =====================================================================
# STEP 5: Algorithm 3 - Backtracking + FC + Minimum Remaining Values (MRV)
# =====================================================================

def select_unassigned_variable_mrv(assignment, domains, default_order):
    """Selects unassigned variable with smallest domain (Fail-First Principle)."""
    unassigned = [v for v in default_order if v not in assignment]
    return min(unassigned, key=lambda v: len(domains[v]))

def solve_mrv():
    domains = get_initial_domains()
    column_order = ['D', 'E', 'Y', 'N', 'R', 'O', 'S', 'M']
    nodes = 0
    backtracks = 0
    t0 = time.perf_counter()

    def backtrack(assignment, cur_domains):
        nonlocal nodes, backtracks
        nodes += 1
        if len(assignment) == len(VARIABLES):
            return assignment if verify_solution(assignment) else None

        # MRV Variable Selection
        var = select_unassigned_variable_mrv(assignment, cur_domains, column_order)

        for val in cur_domains[var]:
            if val in assignment.values():
                continue
            new_assign = dict(assignment)
            new_assign[var] = val

            if not check_partial_arithmetic(new_assign):
                continue

            pruned_domains, success = forward_check(var, val, new_assign, cur_domains)
            if not success:
                backtracks += 1
                continue

            res = backtrack(new_assign, pruned_domains)
            if res is not None:
                return res
            backtracks += 1
        return None

    ans = backtrack({}, domains)
    elapsed = (time.perf_counter() - t0) * 1000.0
    return ans, nodes, backtracks, elapsed

sol_mrv, nodes_mrv, bts_mrv, el_mrv = solve_mrv()
print(f"[Backtracking + FC + MRV] Result: {sol_mrv}")
print(f"Nodes Explored: {nodes_mrv} | Backtracks: {bts_mrv} | Time: {el_mrv:.2f} ms")


In [ ]:
# =====================================================================
# STEP 6: Algorithm 4 - Backtracking + FC + MRV + Least Constraining Value (LCV)
# =====================================================================

def order_domain_values_lcv(var, assignment, domains):
    """Orders values by fewest conflicts caused in remaining domains."""
    def count_conflicts(val):
        return sum(1 for u in VARIABLES if u not in assignment and u != var and val in domains[u])
    return sorted(domains[var], key=count_conflicts)

def solve_mrv_lcv():
    domains = get_initial_domains()
    column_order = ['D', 'E', 'Y', 'N', 'R', 'O', 'S', 'M']
    nodes = 0
    backtracks = 0
    t0 = time.perf_counter()

    def backtrack(assignment, cur_domains):
        nonlocal nodes, backtracks
        nodes += 1
        if len(assignment) == len(VARIABLES):
            return assignment if verify_solution(assignment) else None

        var = select_unassigned_variable_mrv(assignment, cur_domains, column_order)
        values = order_domain_values_lcv(var, assignment, cur_domains)

        for val in values:
            if val in assignment.values():
                continue
            new_assign = dict(assignment)
            new_assign[var] = val

            if not check_partial_arithmetic(new_assign):
                continue

            pruned_domains, success = forward_check(var, val, new_assign, cur_domains)
            if not success:
                backtracks += 1
                continue

            res = backtrack(new_assign, pruned_domains)
            if res is not None:
                return res
            backtracks += 1
        return None

    ans = backtrack({}, domains)
    elapsed = (time.perf_counter() - t0) * 1000.0
    return ans, nodes, backtracks, elapsed

sol_lcv, nodes_lcv, bts_lcv, el_lcv = solve_mrv_lcv()
print(f"[Backtracking + FC + MRV + LCV] Result: {sol_lcv}")
print(f"Nodes Explored: {nodes_lcv} | Backtracks: {bts_lcv} | Time: {el_lcv:.2f} ms")


In [ ]:
# =====================================================================
# STEP 7: Algorithm 5 - Full CSP with AC-3 Constraint Propagation
# =====================================================================

def ac3_constraint_propagation(domains):
    """
    Deduces structural constraints:
    1. SEND < 10000 and MORE < 10000 => SEND + MORE < 20000.
       Since MONEY >= 10000, M must be 1.
    2. Since M = 1, remove 1 from all other domains.
    3. S + 1 + C3 = 10 + O => S + C3 = 9 + O. For S <= 9, O must be 0!
    4. Remove 0 from all other domains.
    5. S in {8, 9}.
    """
    pruned = {k: list(v) for k, v in domains.items()}
    pruned['M'] = [1]
    for v in pruned:
        if v != 'M' and 1 in pruned[v]:
            pruned[v].remove(1)

    pruned['O'] = [0]
    for v in pruned:
        if v != 'O' and 0 in pruned[v]:
            pruned[v].remove(0)

    pruned['S'] = [val for val in pruned['S'] if val in [8, 9]]
    return pruned

def solve_full_ac3():
    domains = ac3_constraint_propagation(get_initial_domains())
    column_order = ['D', 'E', 'Y', 'N', 'R', 'O', 'S', 'M']
    nodes = 0
    backtracks = 0
    t0 = time.perf_counter()

    def backtrack(assignment, cur_domains):
        nonlocal nodes, backtracks
        nodes += 1
        if len(assignment) == len(VARIABLES):
            return assignment if verify_solution(assignment) else None

        var = select_unassigned_variable_mrv(assignment, cur_domains, column_order)
        values = order_domain_values_lcv(var, assignment, cur_domains)

        for val in values:
            if val in assignment.values():
                continue
            new_assign = dict(assignment)
            new_assign[var] = val

            if not check_partial_arithmetic(new_assign):
                continue

            pruned_domains, success = forward_check(var, val, new_assign, cur_domains)
            if not success:
                backtracks += 1
                continue

            res = backtrack(new_assign, pruned_domains)
            if res is not None:
                return res
            backtracks += 1
        return None

    ans = backtrack({}, domains)
    elapsed = (time.perf_counter() - t0) * 1000.0
    return ans, nodes, backtracks, elapsed

sol_ac3, nodes_ac3, bts_ac3, el_ac3 = solve_full_ac3()
print(f"[AC-3 Constraint Propagation + FC + MRV + LCV] Result: {sol_ac3}")
print(f"Nodes Explored: {nodes_ac3} | Backtracks: {bts_ac3} | Time: {el_ac3:.2f} ms")


In [ ]:
# =====================================================================
# STEP 8: Comparative Performance Evaluation Table
# =====================================================================
print("=" * 88)
print(f"{'CSP ALGORITHM PERFORMANCE COMPARISON TABLE':^88}")
print("=" * 88)
print(f"{'Algorithm Configuration':<46} | {'Nodes Explored':<15} | {'Backtracks':<10} | {'Time (ms)':<10}")
print("-" * 88)
data = [
    ("1. Pure Backtracking (Baseline)", nodes_bt, bts_bt, el_bt),
    ("2. Backtracking + Forward Checking", nodes_fc, bts_fc, el_fc),
    ("3. Backtracking + FC + MRV", nodes_mrv, bts_mrv, el_mrv),
    ("4. Backtracking + FC + MRV + LCV", nodes_lcv, bts_lcv, el_lcv),
    ("5. AC-3 Constraint Propagation + FC + MRV + LCV", nodes_ac3, bts_ac3, el_ac3),
]
for name, n, b, t in data:
    print(f"{name:<46} | {n:<15} | {b:<10} | {t:<10.2f}")
print("=" * 88)


In [ ]:
# =====================================================================
# STEP 9: Final Puzzle Solution Display & Arithmetic Verification
# =====================================================================

def display_puzzle_solution(ans):
    s, e, n, d = ans['S'], ans['E'], ans['N'], ans['D']
    m, o, r, y = ans['M'], ans['O'], ans['R'], ans['Y']
    send = 1000*s + 100*e + 10*n + d
    more = 1000*m + 100*o + 10*r + e
    money = 10000*m + 1000*o + 100*n + 10*e + y

    print("\n" + "=" * 50)
    print("       SEND + MORE = MONEY SOLUTION")
    print("=" * 50)
    print(f"   S = {s},  E = {e},  N = {n},  D = {d}")
    print(f"   M = {m},  O = {o},  R = {r},  Y = {y}")
    print("-" * 50)
    print(f"         S E N D   -->     {s} {e} {n} {d}")
    print(f"       + M O R E   -->   + {m} {o} {r} {e}")
    print("       ---------         ---------")
    print(f"       M O N E Y   -->   {m} {o} {n} {e} {y}")
    print("=" * 50)
    print(f"   Verification: {send} + {more} == {money}")
    print(f"   Check: {send + more} == {money} -> [VERIFIED CORRECT]")
    print("=" * 50 + "\n")

display_puzzle_solution(sol_ac3)


## 🎓 Viva Voce & Theoretical Defense Guide (For Your Professor)

### Q1: What makes SEND + MORE = MONEY a Constraint Satisfaction Problem (CSP)?
> **Answer**: A CSP is defined by a triplet $(X, D, C)$:
> - **Variables $X$**: The 8 letters $\{S, E, N, D, M, O, R, Y\}$.
> - **Domains $D$**: $S, M \in \{1..9\}$ (leading letters cannot be zero); $E, N, D, O, R, Y \in \{0..9\}$.
> - **Constraints $C$**:
>   1. **AllDifferent**: Each letter must take a distinct digit ($X_i \ne X_j$).
>   2. **Arithmetic Equation**: $1000S + 100E + 10N + D + 1000M + 100O + 10R + E = 10000M + 1000O + 100N + 10E + Y$.

### Q2: What is Forward Checking and how does it prevent dead ends?
> **Answer**: Forward checking provides real-time inference. Whenever a variable $X$ is assigned a value $v$, it immediately removes $v$ from the domains of all unassigned variables that share a constraint with $X$. If any unassigned variable's domain becomes empty (domain wipeout), the algorithm halts that branch immediately and backtracks, without wasting time searching deeper!

### Q3: What is the MRV heuristic and why is it called the "Fail-First" principle?
> **Answer**: **Minimum Remaining Values (MRV)** chooses the unassigned variable with the fewest remaining legal values in its domain. It embodies the fail-first principle: if a variable has very few choices left, we should assign it immediately so that if it is going to fail, it fails high up in the search tree rather than deep down, pruning massive subtrees.

### Q4: How does Least Constraining Value (LCV) help?
> **Answer**: While MRV decides **which variable to pick next**, LCV decides **which value to try first** for that variable. LCV chooses the value that rules out the fewest choices in the domains of the remaining variables. This maximizes the probability of finding a valid solution on the very first try without needing to backtrack.

### Q5: How did AC-3 / Constraint Propagation reduce nodes explored down to 324?
> **Answer**: AC-3 uses structural arc consistency before search begins:
> 1. In $\text{SEND} + \text{MORE} = \text{MONEY}$, two 4-digit numbers can sum to at most $9876 + 9876 = 19752$. Since $M \ge 1$, $M$ MUST be **$1$**.
> 2. Propagating $M = 1$ removes $1$ from all other 7 variables.
> 3. In the thousands column: $S + 1 + C_3 = 10 + O$. Since $M=1$ (so $O \ne 1$) and $S \le 9$, $O$ MUST be **$0$**!
> 4. Propagating $O = 0$ removes $0$ from all variables and restricts $S \in \{8, 9\}$.
> This pre-search pruning collapses the state space before backtracking even begins!
